# Create AXA Research Fund Awards

Creates AXA Research Fund awards from the fund's own funded-projects database at
https://axafoundation.org/en/science/funded-projects (axa-research.org/funded-projects
redirects there; since 2024 the fund is the Science pillar of the AXA Foundation for
Human Progress). The list is served by the page's own JSON endpoint
(`POST /api/projects/filterByTags[/page/N]`, 10 per page, with the session's XSRF
cookie): project id, title, grant type, research categories/tags and researchers (name,
ORCID, institution, department, country). The project's HTML page (from the site's
sitemap) supplies the lead paragraph used as `description`.

**784 projects** funded since 2008 (Post-Doctoral Fellowships 381, Ph.D 133, AXA Chairs 71,
AXA Projects 66, AXA Awards 42, Joint Research Initiatives 34, Mécénat des Mutuelles 31 —
medical research funded through AXA's mutual insurers — and AXA Outlooks 26).
13 "researcher" records on the site are really grantee organisations (e.g. Fondation
Croix-Rouge française "(FRUP)", OECD, CEPR, Geneva Health Forum); the script keeps them
out of the PI fields (raw column `grantee_organisations`), so those projects have no PI. **Scope:** every grant type on the Science funded-projects list is
kept, including fellowships and PhDs; nothing filtered. The foundation's Nature /
Solidarity / Arts pillars are separate pages and not scraped.

**Not published:** amounts (§6.7 amount check waived), award dates/years (the API gives
a start date for only 28 projects; `start_date` is NULL otherwise, so §6.3's date
criterion is waived), and AXA's grant references (citing works write
e.g. `14-AXA-PDOC-222`, which the site never shows). `funder_award_id` is therefore the
synthetic `AXA-RF-<site project id>` (§2.1.1 step 2; stable site primary key), so no
existing citation stubs collapse.

Lead investigator = first researcher listed; all researchers (with ORCID where given) in
`investigators`. Family names are printed in capitals on the site and title-cased in the
script only when fully upper-case.

**Prerequisites:** run `scripts/local/axa_research_fund_to_s3.py` first. It uploads
`s3://openalex-ingest/awards/axa_research_fund/axa_research_fund_projects.parquet`
(§1.4 shrink guard).

**Funder (Path A, F4320* Crossref-registered):** funder_id `4320321048`, AXA Research Fund,
FR; ror 02zxqxw53 and doi 10.13039/501100001961 read from `openalex.funders.funders`.

**Priority:** `544` (direct funder ingest; higher wins under the 2026-06-20 DESC dedup).

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.axa_research_fund_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/axa_research_fund/axa_research_fund_projects.parquet`;

In [ ]:
%sql
SELECT grant_type, COUNT(*) as n, COUNT(start_date) as has_start_date
FROM openalex.awards.axa_research_fund_raw
GROUP BY 1 ORDER BY n DESC;

In [ ]:
%sql
DESCRIBE openalex.awards.axa_research_fund_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.axa_research_fund_raw LIMIT 5;

## Step 1.6: Funder existence check (Path A)
F4320321048 is a Crossref-registered funder, so it MUST resolve to exactly 1 row in
`openalex.funders.funders`. If 0 rows, STOP and flag.

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.funders.funders
WHERE funder_id = 4320321048;

## Step 2: Create AXA Research Fund Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.axa_research_fund_awards
USING delta
AS
WITH
src_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.funders.funders
    WHERE funder_id = 4320321048  -- AXA Research Fund
),

people AS (
    -- struct field order MUST match openalex_awards_raw: ...orcid, role_start, affiliation.
    SELECT
        funder_award_id,
        TRANSFORM(
            FILTER(
                from_json(people, 'array<struct<given_name:string,family_name:string,orcid:string,institution:string,department:string,country:string>>'),
                p -> p.family_name IS NOT NULL AND TRIM(p.family_name) != ''
            ),
            p -> named_struct(
                'given_name', NULLIF(TRIM(p.given_name), ''),
                'family_name', NULLIF(TRIM(p.family_name), ''),
                'orcid', CASE WHEN p.orcid RLIKE '^[0-9]{4}-[0-9]{4}-[0-9]{4}-[0-9]{3}[0-9X]$'
                              THEN CONCAT('https://orcid.org/', p.orcid) END,
                'role_start', CAST(NULL AS DATE),
                'affiliation', named_struct(
                    'name', NULLIF(TRIM(p.institution), ''),
                    'country', NULLIF(TRIM(p.country), ''),
                    'ids', CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>)
                )
            )
        ) AS investigators
    FROM openalex.awards.axa_research_fund_raw
)

SELECT
    abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000 as id,
    NULLIF(TRIM(g.title), '') as display_name,
    COALESCE(NULLIF(TRIM(g.description), ''), NULLIF(TRIM(g.excerpt), '')) as description,
    f.funder_id,
    TRIM(g.funder_award_id) as funder_award_id,
    CAST(NULL AS DOUBLE) as amount,
    CAST(NULL AS STRING) as currency,
    struct(
        CONCAT('https://openalex.org/F', f.funder_id) as id,
        f.display_name,
        f.ror_id,
        f.doi
    ) as funder,
    CASE
        WHEN lower(g.grant_type) LIKE 'post-doctoral%' THEN 'fellowship'
        WHEN lower(g.grant_type) IN ('ph.d', 'phd', 'ph.d.') THEN 'studentship'
        WHEN lower(g.grant_type) LIKE '%m_c_nat%' THEN 'grant'
        ELSE 'research'
    END as funding_type,
    NULLIF(TRIM(g.grant_type), '') as funder_scheme,
    'axa_research_fund_projects' as provenance,
    COALESCE(TRY_TO_DATE(g.start_date, 'yyyy-MM-dd'), TRY_TO_DATE(SUBSTR(g.start_date, 1, 10), 'yyyy-MM-dd')) as start_date,
    CAST(NULL AS DATE) as end_date,
    YEAR(COALESCE(TRY_TO_DATE(g.start_date, 'yyyy-MM-dd'), TRY_TO_DATE(SUBSTR(g.start_date, 1, 10), 'yyyy-MM-dd'))) as start_year,
    CAST(NULL AS INT) as end_year,
    -- try_element_at: safe on empty arrays (§2.3)
    try_element_at(p.investigators, 1) as lead_investigator,
    try_element_at(p.investigators, 2) as co_lead_investigator,
    CASE WHEN size(p.investigators) > 0 THEN p.investigators END as investigators,
    g.landing_page_url as landing_page_url,
    CAST(NULL AS STRING) as doi,
    concat('https://api.openalex.org/works?filter=awards.id:G',
           abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000) as works_api_url,
    current_timestamp() as created_date,
    current_timestamp() as updated_date
FROM openalex.awards.axa_research_fund_raw g
JOIN people p ON p.funder_award_id = g.funder_award_id
CROSS JOIN src_funder f
WHERE g.funder_award_id IS NOT NULL AND TRIM(g.funder_award_id) != '';

In [ ]:
%sql
-- §2.1.1 shape check: synthetic AXA-RF-<site project id>; expect bad_shape = 0, duplicate_ids = 0.
SELECT
    COUNT(*) as total,
    SUM(CASE WHEN funder_award_id RLIKE '^AXA-RF-[0-9]+$' THEN 0 ELSE 1 END) as bad_shape,
    COUNT(*) - COUNT(DISTINCT id) as duplicate_ids
FROM openalex.awards.axa_research_fund_awards;

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'axa_research_fund_projects' AND priority = 544;

-- Insert into openalex_awards_raw with priority.
-- Priority 544: direct-from-funder ingest of the AXA Research Fund project database.
-- Higher wins under the 2026-06-20 DESC dedup (oxjob #500).
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    544 as priority
FROM openalex.awards.axa_research_fund_awards;

## Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total_awards, COUNT(DISTINCT id) as distinct_ids
FROM openalex.awards.axa_research_fund_awards;

In [ ]:
%sql
SELECT id, display_name, funder_award_id, funder_scheme, funding_type, start_year,
       lead_investigator.given_name, lead_investigator.family_name, lead_investigator.orcid,
       lead_investigator.affiliation.name, lead_investigator.affiliation.country,
       size(investigators) as n_investigators
FROM openalex.awards.axa_research_fund_awards LIMIT 20;

In [ ]:
%sql
SELECT funder_scheme, funding_type, COUNT(*) as cnt
FROM openalex.awards.axa_research_fund_awards
GROUP BY 1, 2 ORDER BY cnt DESC;

In [ ]:
%sql
SELECT start_year, COUNT(*) as cnt
FROM openalex.awards.axa_research_fund_awards
GROUP BY start_year ORDER BY start_year;

In [ ]:
%sql
-- §6.4a frequency check: PI top-20 must be a real long-tail.
SELECT lead_investigator.given_name AS given, lead_investigator.family_name AS family, COUNT(*) AS n
FROM openalex.awards.axa_research_fund_awards
GROUP BY 1, 2 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
-- §6.4a title frequency check.
SELECT display_name, COUNT(*) AS n
FROM openalex.awards.axa_research_fund_awards
GROUP BY 1 ORDER BY n DESC LIMIT 10;

In [ ]:
%sql
-- §6.3 coverage (expect ~100% title/PI/institution; amount 0% because AXA publishes none).
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(description) as has_description,
    COUNT(amount) as has_amount,
    COUNT(start_date) as has_start_date,
    COUNT(lead_investigator.family_name) as has_pi_name,
    COUNT(lead_investigator.orcid) as has_pi_orcid,
    COUNT(lead_investigator.affiliation.name) as has_institution
FROM openalex.awards.axa_research_fund_awards;

In [ ]:
%sql
-- Confirm rows reached the shared raw table at the assigned priority (§6.8).
SELECT provenance, priority, COUNT(*) as n
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'axa_research_fund_projects'
GROUP BY provenance, priority;